# GBD Vietnam Epidemiological Transition Analysis, 1990-2023

Target: *Lancet Regional Health - Western Pacific*.

This notebook is generated from `analysis.py` and executes the full
matplotlib pipeline top to bottom.
Core metrics follow the GBD 2023 convention: age-standardized DALY
composition shares and CMNN/NCD ratio (replacing the earlier ad-hoc
indices).


## Imports & configuration

In [1]:
"""
GBD Vietnam Epidemiological Transition Analysis, 1990-2023.
Target: Lancet Regional Health - Western Pacific.

Sections
--------
1. Load & Clean Data
2. Core Metrics (cause composition shares, CMNN/NCD ratio)
3. Trend Analysis (Joinpoint via `ruptures` + APC + AAPC)
   Kim HJ et al. Stat Med 2000;19:335-351.
   Clegg LX et al. Stat Med 2009 (AAPC delta-method CI).
4. Decomposition (Das Gupta 1993, Standardization and Decomposition of Rates)
5. SEA Comparison (NCD share + SDI-expected ratio;
   Lim SS et al. Lancet 2018;392:2091-138)
6. Figures (publication-ready)
7. Summary Table
"""

import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import ruptures as rpt

warnings.filterwarnings("ignore")

# =============================================================================
# Paths and global config
# =============================================================================

# Resolve project root relative to this file so the script is portable.
try:
    ROOT = Path(__file__).resolve().parent
except NameError:
    ROOT = Path.cwd()
RAW = ROOT / "data" / "raw"
PROC = ROOT / "data" / "processed"
FIG = ROOT / "figures"
TAB = ROOT / "tables"
for p in (PROC, FIG, TAB):
    p.mkdir(parents=True, exist_ok=True)

YEAR_MIN, YEAR_MAX = 1990, 2023

# 11 SEA countries used in the paper
SEA_COUNTRIES = [
    "Viet Nam", "Thailand", "Indonesia", "Philippines", "Malaysia",
    "Myanmar", "Cambodia", "Lao People's Democratic Republic",
    "Singapore", "Brunei Darussalam", "Timor-Leste",
]

# Display names
COUNTRY_DISPLAY = {
    "Viet Nam": "Vietnam",
    "Lao People's Democratic Republic": "Lao PDR",
    "Brunei Darussalam": "Brunei",
}

# Okabe-Ito colour-blind-friendly palette
PALETTE = {
    "CMNN": "#E69F00",
    "NCD": "#0072B2",
    "Injuries": "#009E73",
    "Vietnam": "#D55E00",
    "neutral": "#999999",
    "accent": "#CC79A7",
    "highlight": "#56B4E9",
}

# Publication figure defaults
mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "DejaVu Sans", "Liberation Sans"],
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "legend.fontsize": 9,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "savefig.dpi": 300,
    "figure.dpi": 150,
})

# mm -> inches, column widths
MM = 1 / 25.4
W1 = 88 * MM
W2 = 180 * MM

# Canonical GBD labels
CMNN_LABEL = "Communicable, maternal, neonatal, and nutritional diseases"
NCD_LABEL = "Non-communicable diseases"
INJ_LABEL = "Injuries"
DALY_LABEL = "DALYs (Disability-Adjusted Life Years)"


# =============================================================================
# Utility functions
# =============================================================================

def snake_case(df):
    df = df.copy()
    df.columns = [
        c.strip().lower().replace(" ", "_").replace("-", "_")
        for c in df.columns
    ]
    return df


def save_fig(fig, name):
    for ext in ("png", "svg"):
        fig.savefig(FIG / f"{name}.{ext}", dpi=300, bbox_inches="tight")
    print(f"  saved figures/{name}.png + .svg")


def shade_ui(ax, x, lower, upper, color, alpha=0.18, label=None):
    ax.fill_between(x, lower, upper, color=color, alpha=alpha,
                    linewidth=0, label=label)


## SECTION 1 - LOAD & CLEAN DATA

In [2]:
# =============================================================================
# SECTION 1 - LOAD & CLEAN DATA
# =============================================================================

def section1_load_clean():
    print("=" * 70)
    print("SECTION 1 - Load & Clean Data")
    print("=" * 70)

    # Level-1 cause groups: SEA countries + Timor concatenated first,
    # then standardized, filtered, and tagged.
    q1a = pd.read_csv(RAW / "query1_level1.csv")
    q1b = pd.read_csv(RAW / "query1_level1_timor.csv")
    q1 = pd.concat([q1a, q1b], ignore_index=True)
    q1 = snake_case(q1)
    print(f"  query1 (level-1 SEA+Timor): {len(q1):,} rows, "
          f"{q1['location_name'].nunique()} locations")

    q1 = q1[q1["location_name"].isin(SEA_COUNTRIES)].copy()
    q1 = q1[(q1["year"] >= YEAR_MIN) & (q1["year"] <= YEAR_MAX)].copy()
    q1["cause_group"] = q1["cause_name"].map({
        CMNN_LABEL: "CMNN",
        NCD_LABEL: "NCD",
        INJ_LABEL: "Injuries",
        "All causes": "All",
    })

    # Vietnam disease-detail tables
    q2a = snake_case(pd.read_csv(RAW / "query2a_cmnn.csv"))
    q2b = snake_case(pd.read_csv(RAW / "query2b_ncd.csv"))
    q2a = q2a[(q2a["year"] >= YEAR_MIN) & (q2a["year"] <= YEAR_MAX)].copy()
    q2b = q2b[(q2b["year"] >= YEAR_MIN) & (q2b["year"] <= YEAR_MAX)].copy()
    print(f"  query2a (Vietnam CMNN detail): {len(q2a):,} rows, "
          f"{q2a['cause_name'].nunique()} causes")
    print(f"  query2b (Vietnam NCD detail):  {len(q2b):,} rows, "
          f"{q2b['cause_name'].nunique()} causes")

    # Age-specific Vietnam data
    q3 = snake_case(pd.read_csv(RAW / "query3_age.csv"))
    q3 = q3[(q3["year"] >= YEAR_MIN) & (q3["year"] <= YEAR_MAX)].copy()
    print(f"  query3 (Vietnam age-specific): {len(q3):,} rows, "
          f"ages={q3['age_name'].nunique()}")

    # Population
    q4 = snake_case(pd.read_csv(RAW / "query4_pop.csv"))
    q4 = q4[(q4["year"] >= YEAR_MIN) & (q4["year"] <= YEAR_MAX)].copy()
    print(f"  query4 (Vietnam population): {len(q4):,} rows")

    # SDI (rename to match conventional GBD column names)
    sdi = snake_case(pd.read_csv(RAW / "SDI.csv"))
    sdi = sdi.rename(columns={"year_id": "year", "mean_value": "sdi"})
    sdi = sdi[sdi["location_name"].isin(SEA_COUNTRIES)].copy()
    sdi = sdi[(sdi["year"] >= YEAR_MIN) & (sdi["year"] <= YEAR_MAX)].copy()
    print(f"  SDI: {len(sdi):,} rows, locations={sdi['location_name'].nunique()}")

    # Cause hierarchy (for reference)
    hier = snake_case(pd.read_excel(RAW / "hierarchy.XLSX"))

    # Data-quality report
    print("\n  --- Data quality ---")
    for name, df in [("q1", q1), ("q2a", q2a), ("q2b", q2b),
                     ("q3", q3), ("q4", q4), ("sdi", sdi)]:
        na = df.isna().sum().sum()
        yr_lo, yr_hi = df["year"].min(), df["year"].max()
        locs = df["location_name"].nunique() if "location_name" in df else "n/a"
        print(f"  {name}: missing={na}, year=[{yr_lo}-{yr_hi}], locations={locs}")

    # Split age-standardized vs all-ages rows for later use
    def split_ages(df, label):
        asr = df[df["age_name"] == "Age-standardized"].copy()
        allages = df[df["age_name"] == "All ages"].copy()
        print(f"  {label}: age-std={len(asr):,} rows, all-ages={len(allages):,} rows")
        return asr, allages

    q1_asr, q1_all = split_ages(q1, "q1")
    q2a_asr, _ = split_ages(q2a, "q2a")
    q2b_asr, _ = split_ages(q2b, "q2b")

    # Save cleaned files
    q1.to_csv(PROC / "level1_sea_1990_2023.csv", index=False)
    q1_asr.to_csv(PROC / "level1_sea_asr.csv", index=False)
    q1_all.to_csv(PROC / "level1_sea_allages.csv", index=False)
    q2a.to_csv(PROC / "vietnam_cmnn_detail.csv", index=False)
    q2b.to_csv(PROC / "vietnam_ncd_detail.csv", index=False)
    q3.to_csv(PROC / "vietnam_age_specific.csv", index=False)
    q4.to_csv(PROC / "vietnam_population.csv", index=False)
    sdi.to_csv(PROC / "sdi_sea.csv", index=False)
    print("  --> cleaned files written to data/processed/")

    return dict(
        q1=q1, q1_asr=q1_asr, q1_all=q1_all,
        q2a=q2a, q2b=q2b, q3=q3, q4=q4, sdi=sdi, hier=hier,
    )


## SECTION 2 - CORE METRICS (cause composition shares)

In [3]:
# =============================================================================
# SECTION 2 - CORE METRICS (cause composition shares)
# =============================================================================
# Uses the standard GBD-convention cause-composition metrics: CMNN / NCD /
# Injuries share of total age-standardized DALYs, plus CMNN/NCD ratio.

def section2_core_metrics(data):
    print("\n" + "=" * 70)
    print("SECTION 2 - Core Metrics (cause composition shares)")
    print("=" * 70)

    q1_asr = data["q1_asr"]
    daly = q1_asr[
        (q1_asr["measure_name"] == DALY_LABEL)
        & (q1_asr["metric_name"] == "Rate")
        & (q1_asr["sex_name"] == "Both")
        & (q1_asr["cause_group"].isin(["CMNN", "NCD", "Injuries"]))
    ].copy()

    pvt = daly.pivot_table(
        index=["location_name", "year"],
        columns="cause_group",
        values="val",
    ).reset_index()
    pvt.columns.name = None
    for c in ("CMNN", "NCD", "Injuries"):
        if c not in pvt.columns:
            pvt[c] = np.nan

    total = pvt[["CMNN", "NCD", "Injuries"]].sum(axis=1)
    pvt["cmnn_share_pct"] = pvt["CMNN"] / total * 100
    pvt["ncd_share_pct"] = pvt["NCD"] / total * 100
    pvt["injuries_share_pct"] = pvt["Injuries"] / total * 100
    pvt["cmnn_ncd_ratio"] = pvt["CMNN"] / pvt["NCD"]

    pvt["country"] = pvt["location_name"].map(
        lambda x: COUNTRY_DISPLAY.get(x, x)
    )
    pvt.to_csv(TAB / "metrics.csv", index=False)

    print("  NCD share of total DALYs in 2023 by country:")
    latest = (pvt[pvt["year"] == 2023]
              [["country", "cmnn_share_pct", "ncd_share_pct",
                "injuries_share_pct", "cmnn_ncd_ratio"]]
              .sort_values("ncd_share_pct", ascending=False))
    print(latest.to_string(index=False, float_format="%.3f"))
    print("  --> tables/metrics.csv")
    return pvt


## SECTION 3 - JOINPOINT / APC / AAPC

In [4]:
# =============================================================================
# SECTION 3 - JOINPOINT / APC / AAPC
# =============================================================================

def joinpoint_apc_aapc(years, y, max_bkps=3):
    """
    Piecewise-linear joinpoint regression on log(y) vs year using
    ruptures.Dynp with RSS cost, selecting the number of breakpoints
    (0..max_bkps) by BIC. Returns joinpoints, per-segment APC and overall AAPC.
    """
    years = np.asarray(years, dtype=float)
    y = np.asarray(y, dtype=float)
    mask = y > 0
    years = years[mask]
    y = y[mask]
    n = len(years)
    if n < 4:
        return None
    log_y = np.log(y)

    # Detrend against linear fit first so change-point detection picks slope
    # changes rather than absolute level. We then refit per segment on log_y.
    X_full = np.vstack([years, np.ones_like(years)]).T
    beta_full, *_ = np.linalg.lstsq(X_full, log_y, rcond=None)
    resid = log_y - X_full @ beta_full

    def _ssr_partition(bkps_idx):
        # bkps_idx: list of right-endpoints (exclusive), last is n
        ssr = 0.0
        prev = 0
        for b in bkps_idx:
            xs = years[prev:b]
            ys = log_y[prev:b]
            if len(xs) < 2:
                ssr += 0.0
            else:
                Xs = np.vstack([xs, np.ones_like(xs)]).T
                bts, *_ = np.linalg.lstsq(Xs, ys, rcond=None)
                ssr += float(((ys - Xs @ bts) ** 2).sum())
            prev = b
        return ssr

    # BIC across k = 0..max_bkps breakpoints
    best = {"bic": np.inf, "bkps_idx": [n], "ssr": None, "k": 0}
    for k in range(0, max_bkps + 1):
        # Need at least 2 points per segment
        if (k + 1) * 2 > n:
            continue
        try:
            if k == 0:
                bkps = [n]
            else:
                algo = rpt.Dynp(model="l2", min_size=2, jump=1).fit(resid)
                bkps = algo.predict(n_bkps=k)
        except Exception:
            continue
        ssr = _ssr_partition(bkps)
        if ssr <= 0:
            ssr = 1e-12
        # 2 params per segment + k breakpoint positions
        p = 2 * (k + 1) + k
        bic = n * np.log(ssr / n) + p * np.log(n)
        if bic < best["bic"]:
            best.update({"bic": bic, "bkps_idx": bkps, "ssr": ssr, "k": k})

    # Build segments
    segments = []
    prev = 0
    for b in best["bkps_idx"]:
        xs = years[prev:b]
        ys = log_y[prev:b]
        if len(xs) < 2:
            prev = b
            continue
        Xs = np.vstack([xs, np.ones_like(xs)]).T
        bts, *_ = np.linalg.lstsq(Xs, ys, rcond=None)
        slope = float(bts[0])
        apc = (np.exp(slope) - 1) * 100
        segments.append({
            "start_year": int(xs[0]),
            "end_year": int(xs[-1]),
            "slope_logyear": slope,
            "intercept": float(bts[1]),
            "apc_pct": apc,
            "n_years": int(len(xs)),
        })
        prev = b

    # Interior joinpoints are the years at segment boundaries
    joinpoints = [s["end_year"] for s in segments[:-1]]

    # AAPC = weighted log-mean of (1 + APC/100), weights = segment span in years
    weights = np.array([max(s["n_years"] - 1, 1) for s in segments], dtype=float)
    if weights.sum() <= 0:
        aapc = segments[0]["apc_pct"] if segments else np.nan
    else:
        log_mean = np.sum(
            np.log1p(np.array([s["apc_pct"] / 100 for s in segments])) * weights
        ) / weights.sum()
        aapc = (np.exp(log_mean) - 1) * 100

    return {
        "joinpoints": joinpoints,
        "segments": segments,
        "aapc_pct": float(aapc),
        "n_obs": int(n),
        "bic_k": int(best["k"]),
    }


def section3_trends(data):
    print("\n" + "=" * 70)
    print("SECTION 3 - Trend Analysis (Joinpoint + APC + AAPC)")
    print("=" * 70)

    q1_asr = data["q1_asr"]
    daly = q1_asr[
        (q1_asr["measure_name"] == DALY_LABEL)
        & (q1_asr["metric_name"] == "Rate")
        & (q1_asr["sex_name"] == "Both")
        & (q1_asr["cause_group"].isin(["CMNN", "NCD", "Injuries"]))
    ].copy()

    rows = []
    for country in SEA_COUNTRIES:
        sub_c = daly[daly["location_name"] == country]
        for group in ["CMNN", "NCD", "Injuries"]:
            sub = sub_c[sub_c["cause_group"] == group].sort_values("year")
            if len(sub) < 4:
                continue
            res = joinpoint_apc_aapc(sub["year"].values, sub["val"].values,
                                     max_bkps=3)
            if res is None:
                continue
            for seg in res["segments"]:
                rows.append({
                    "country": COUNTRY_DISPLAY.get(country, country),
                    "location_name": country,
                    "cause_group": group,
                    "segment_start": seg["start_year"],
                    "segment_end": seg["end_year"],
                    "apc_pct": seg["apc_pct"],
                    "aapc_pct_1990_2023": res["aapc_pct"],
                    "joinpoints": ";".join(str(j) for j in res["joinpoints"]),
                    "n_years_segment": seg["n_years"],
                    "n_breakpoints": res["bic_k"],
                })
    tr = pd.DataFrame(rows)
    tr.to_csv(TAB / "trend_results.csv", index=False)

    vn = (tr[tr["country"] == "Vietnam"]
          .drop_duplicates(subset=["cause_group"], keep="first")
          [["cause_group", "aapc_pct_1990_2023", "joinpoints", "n_breakpoints"]])
    print("  Vietnam AAPC 1990-2023 (age-std DALY rate):")
    print(vn.to_string(index=False, float_format="%.2f"))
    print(f"  --> tables/trend_results.csv ({len(tr)} segment rows)")
    return tr


## SECTION 4 - DAS GUPTA DECOMPOSITION

In [5]:
# =============================================================================
# SECTION 4 - DAS GUPTA DECOMPOSITION
# =============================================================================

def das_gupta_decomposition(pop_t0, rate_t0, pop_t1, rate_t1):
    """
    Symmetric three-factor Das Gupta decomposition of the change in total
    events D = P * sum(s * r) between two time points:

        dD = (population size effect)
           + (age structure effect)
           + (age-specific rate effect)

    pop_*:  age-specific populations
    rate_*: age-specific rates per person
    """
    P0, P1 = pop_t0.sum(), pop_t1.sum()
    s0 = pop_t0 / P0
    s1 = pop_t1 / P1
    r0, r1 = rate_t0, rate_t1

    size = ((P1 - P0) / 2) * (np.sum(s0 * r0) + np.sum(s1 * r1))
    age_struct = ((P0 + P1) / 2) * np.sum((s1 - s0) * (r0 + r1) / 2)
    rate_eff = ((P0 + P1) / 2) * np.sum((s0 + s1) / 2 * (r1 - r0))

    total = size + age_struct + rate_eff
    observed = np.sum(pop_t1 * r1) - np.sum(pop_t0 * r0)
    return {
        "pop_size": size,
        "age_structure": age_struct,
        "age_rate": rate_eff,
        "total_decomp": total,
        "observed_change": observed,
        "residual": observed - total,
    }


def section4_decomposition(data):
    print("\n" + "=" * 70)
    print("SECTION 4 - Das Gupta Decomposition (1990 -> 2023)")
    print("=" * 70)

    q3 = data["q3"]
    q4 = data["q4"]

    age_groups = [
        "<5 years", "5-9 years", "10-14 years", "15-19 years", "20-24 years",
        "25-29 years", "30-34 years", "35-39 years", "40-44 years",
        "45-49 years", "50-54 years", "55-59 years", "60-64 years",
        "65-69 years", "70-74 years", "75-79 years", "80+ years",
    ]

    pop = q4[
        (q4["sex_name"] == "Both")
        & (q4["age_name"].isin(age_groups))
        & (q4["measure_name"] == "Population")
    ].copy()
    pop_wide = pop.pivot_table(
        index="age_name", columns="year", values="val"
    ).reindex(age_groups)

    daly_age = q3[
        (q3["measure_name"] == DALY_LABEL)
        & (q3["metric_name"] == "Rate")
        & (q3["sex_name"] == "Both")
        & (q3["age_name"].isin(age_groups))
    ].copy()
    daly_age["cause_group"] = daly_age["cause_name"].map(
        {CMNN_LABEL: "CMNN", NCD_LABEL: "NCD"}
    )
    daly_age = daly_age[daly_age["cause_group"].notna()]

    rows = []
    for cg in ["CMNN", "NCD"]:
        sub = daly_age[daly_age["cause_group"] == cg]
        rate_wide = sub.pivot_table(
            index="age_name", columns="year", values="val"
        ).reindex(age_groups)
        # GBD rates are per 100k — convert to per-person for DALY totals
        r1990 = (rate_wide[1990] / 1e5).values
        r2023 = (rate_wide[2023] / 1e5).values
        p1990 = pop_wide[1990].values
        p2023 = pop_wide[2023].values
        decomp = das_gupta_decomposition(p1990, r1990, p2023, r2023)
        rows.append({"cause_group": cg, **decomp})

    tbl = pd.DataFrame(rows)
    tbl["direction"] = np.where(tbl["observed_change"] >= 0, "increase", "decrease")
    tbl.to_csv(TAB / "decomposition_results.csv", index=False)

    print("  Observed vs decomposed change in total DALYs (Vietnam):")
    show = tbl[["cause_group", "pop_size", "age_structure",
                "age_rate", "total_decomp", "observed_change", "direction"]]
    print(show.to_string(index=False, float_format="%.0f"))
    print("  --> tables/decomposition_results.csv")
    return tbl


## SECTION 5 - SEA COMPARISON

In [6]:
# =============================================================================
# SECTION 5 - SEA COMPARISON
# =============================================================================

def section5_sea(data, metrics):
    print("\n" + "=" * 70)
    print("SECTION 5 - SEA Comparison")
    print("=" * 70)

    snapshot_years = [1990, 2000, 2010, 2023]
    snap = (metrics[metrics["year"].isin(snapshot_years)]
            [["country", "year", "ncd_share_pct"]]
            .pivot(index="country", columns="year",
                   values="ncd_share_pct"))
    snap.columns = [f"ncd_share_{y}" for y in snap.columns]
    snap["delta_ncd_share_pp"] = (
        snap["ncd_share_2023"] - snap["ncd_share_1990"]
    )

    # SDI-expected NCD share (quadratic fit, Vietnam excluded). Annotates
    # Vietnam's observed/expected ratio for 2023.
    # Lim SS et al. Lancet 2018;392:2091-138.
    sdi = data["sdi"]
    m23 = metrics[metrics["year"] == 2023].merge(
        sdi[sdi["year"] == 2023][["location_name", "sdi"]],
        on="location_name", how="left",
    )
    fit = m23[(m23["country"] != "Vietnam")
              & m23["sdi"].notna() & m23["ncd_share_pct"].notna()]
    ratios = {}
    if len(fit) >= 3:
        coef = np.polyfit(fit["sdi"].values,
                          fit["ncd_share_pct"].values, 2)
        for _, row in m23.iterrows():
            if pd.isna(row["sdi"]):
                continue
            exp = float(np.polyval(coef, row["sdi"]))
            obs = float(row["ncd_share_pct"])
            ratios[row["country"]] = obs / exp if exp else np.nan
    snap["obs_vs_expected_ratio_2023"] = (
        snap.index.map(ratios.get).astype(float)
    )

    sea = snap.reset_index().sort_values("ncd_share_2023", ascending=False)
    sea.to_csv(TAB / "sea_comparison.csv", index=False)

    print("  SEA countries ranked by NCD share of total DALYs, 2023:")
    print(sea[["country", "ncd_share_1990", "ncd_share_2023",
               "delta_ncd_share_pp", "obs_vs_expected_ratio_2023"]]
          .to_string(index=False, float_format="%.3f"))
    print("  --> tables/sea_comparison.csv")
    return sea


## SECTION 6 - FIGURES

In [7]:
# =============================================================================
# SECTION 6 - FIGURES
# =============================================================================

def figure_1(data, metrics):
    """2x2 overview panel for Vietnam."""
    print("\n  Figure 1 - 2x2 Overview Panel (Vietnam)")
    q1 = data["q1"]
    vn = q1[
        (q1["location_name"] == "Viet Nam")
        & (q1["measure_name"] == DALY_LABEL)
        & (q1["sex_name"] == "Both")
    ].copy()

    num_aa = vn[(vn["metric_name"] == "Number") & (vn["age_name"] == "All ages")]
    rate_as = vn[(vn["metric_name"] == "Rate") & (vn["age_name"] == "Age-standardized")]

    num_pvt = num_aa.pivot_table(index="year", columns="cause_group", values="val")
    rate_pvt = rate_as.pivot_table(index="year", columns="cause_group", values="val")
    rate_lo = rate_as.pivot_table(index="year", columns="cause_group", values="lower")
    rate_hi = rate_as.pivot_table(index="year", columns="cause_group", values="upper")
    for df in (num_pvt, rate_pvt, rate_lo, rate_hi):
        for col in ["CMNN", "NCD", "Injuries"]:
            if col not in df.columns:
                df[col] = np.nan

    years = num_pvt.index.values
    cmnn = num_pvt["CMNN"].values
    ncd = num_pvt["NCD"].values
    inj = num_pvt["Injuries"].values
    total = cmnn + ncd + inj

    fig, axes = plt.subplots(2, 2, figsize=(W2, W2 * 0.70))

    # Panel A: absolute DALYs stacked
    ax = axes[0, 0]
    ax.stackplot(years, cmnn / 1e6, ncd / 1e6, inj / 1e6,
                 labels=["CMNN", "NCD", "Injuries"],
                 colors=[PALETTE["CMNN"], PALETTE["NCD"], PALETTE["Injuries"]],
                 alpha=0.85)
    ax.set_title("A. Absolute DALYs by cause group, Vietnam")
    ax.set_xlabel("Year")
    ax.set_ylabel("DALYs (millions)")
    ax.legend(loc="upper right", frameon=False)
    ax.set_xlim(YEAR_MIN, YEAR_MAX)

    # Panel B: % contribution stacked
    ax = axes[0, 1]
    ax.stackplot(years, cmnn / total * 100, ncd / total * 100, inj / total * 100,
                 labels=["CMNN", "NCD", "Injuries"],
                 colors=[PALETTE["CMNN"], PALETTE["NCD"], PALETTE["Injuries"]],
                 alpha=0.85)
    ax.set_title("B. % contribution of each group, Vietnam")
    ax.set_xlabel("Year")
    ax.set_ylabel("% of total DALYs")
    ax.set_ylim(0, 100)
    ax.set_xlim(YEAR_MIN, YEAR_MAX)
    ax.legend(loc="center right", frameon=False)

    # Panel C: Age-std DALY rates with UI
    ax = axes[1, 0]
    for g, color in [("CMNN", PALETTE["CMNN"]), ("NCD", PALETTE["NCD"]),
                     ("Injuries", PALETTE["Injuries"])]:
        ax.plot(rate_pvt.index, rate_pvt[g], color=color, lw=2, label=g)
        shade_ui(ax, rate_pvt.index, rate_lo[g], rate_hi[g], color=color)
    ax.set_title("C. Age-standardized DALY rate, Vietnam")
    ax.set_xlabel("Year")
    ax.set_ylabel("DALYs per 100,000")
    ax.legend(frameon=False)
    ax.set_xlim(YEAR_MIN, YEAR_MAX)

    # Panel D: NCD share of total DALYs
    # share = NCD age-std DALYs / (CMNN + NCD + Injuries) x 100.
    ax = axes[1, 1]
    vn_metrics = metrics[metrics["country"] == "Vietnam"].sort_values("year")
    ax.plot(vn_metrics["year"], vn_metrics["ncd_share_pct"],
            color=PALETTE["NCD"], lw=2.2, label="NCD share (%), Vietnam")
    ax.axhline(50, color=PALETTE["neutral"], ls=":", lw=1, label="50% reference")
    ax.set_ylim(0, 100)
    ax.set_title("D. NCD share of total DALYs, Vietnam")
    ax.set_xlabel("Year")
    ax.set_ylabel("NCD share of total DALYs (%)")
    ax.legend(loc="lower right", frameon=False)
    ax.set_xlim(YEAR_MIN, YEAR_MAX)

    fig.tight_layout()
    save_fig(fig, "figure1_overview")
    plt.close(fig)


def figure_2(data):
    """Heatmap of top-15 Vietnam causes (age-std DALY rate)."""
    print("  Figure 2 - Top-15 cause heatmap (Vietnam)")
    det = pd.concat([data["q2a"], data["q2b"]], ignore_index=True)
    det = det[
        (det["measure_name"] == DALY_LABEL)
        & (det["metric_name"] == "Rate")
        & (det["age_name"] == "Age-standardized")
        & (det["sex_name"] == "Both")
        & (~det["cause_name"].isin([CMNN_LABEL, NCD_LABEL]))
    ].copy()

    latest = det[det["year"] == 2023]
    top15 = (latest.sort_values("val", ascending=False)
             .drop_duplicates("cause_name")["cause_name"].head(15).tolist())

    years_plot = [1990, 1995, 2000, 2005, 2010, 2015, 2020, 2023]
    mat = (det[det["cause_name"].isin(top15) & det["year"].isin(years_plot)]
           .pivot_table(index="cause_name", columns="year", values="val")
           .reindex(top15))

    rel = mat.div(mat[1990], axis=0)
    fig, ax = plt.subplots(figsize=(W2, W2 * 0.62))
    cmap = plt.get_cmap("RdBu_r")
    vmax = float(np.nanmax(np.abs(rel - 1)))
    im = ax.imshow(rel.values, aspect="auto", cmap=cmap,
                   vmin=1 - vmax, vmax=1 + vmax, interpolation="nearest")
    ax.set_xticks(range(len(years_plot)))
    ax.set_xticklabels(years_plot)
    def wrap(s, maxw=40):
        return s if len(s) < maxw else s[:maxw - 1] + "..."
    ax.set_yticks(range(len(top15)))
    ax.set_yticklabels([wrap(c) for c in top15])
    ax.set_title("Age-standardized DALY rate relative to 1990 "
                 "(Vietnam, top 15 causes by 2023 rate)")
    for i in range(len(top15)):
        for j in range(len(years_plot)):
            v = mat.iloc[i, j]
            if not np.isnan(v):
                ax.text(j, i, f"{v:.0f}", ha="center", va="center",
                        fontsize=7, color="black")
    cbar = fig.colorbar(im, ax=ax, shrink=0.8, pad=0.02)
    cbar.set_label("Rate ratio vs 1990")
    fig.tight_layout()
    save_fig(fig, "figure2_heatmap")
    plt.close(fig)


def figure_3(decomp):
    """Waterfall decomposition chart."""
    print("  Figure 3 - Decomposition waterfall")
    fig, axes = plt.subplots(1, 2, figsize=(W2, W2 * 0.45))

    for ax, cg in zip(axes, ["NCD", "CMNN"]):
        row = decomp[decomp["cause_group"] == cg].iloc[0]
        components = [
            ("Population size", row["pop_size"]),
            ("Age structure", row["age_structure"]),
            ("Age-specific rate", row["age_rate"]),
        ]
        values = [v / 1e6 for _, v in components]
        total = sum(values)
        labels = [n for n, _ in components]

        running = 0
        bottom, heights, colors = [], [], []
        for v in values:
            if v >= 0:
                bottom.append(running)
                heights.append(v)
                colors.append(PALETTE["Injuries"])
            else:
                bottom.append(running + v)
                heights.append(-v)
                colors.append(PALETTE["CMNN"])
            running += v

        ax.bar(labels, heights, bottom=bottom, color=colors, alpha=0.85,
               edgecolor="black", lw=0.5)
        ax.bar("Net change", abs(total), bottom=0 if total >= 0 else total,
               color=PALETTE["NCD"] if cg == "NCD" else PALETTE["CMNN"],
               alpha=0.9, edgecolor="black", lw=0.5)
        for i, v in enumerate(values):
            top_y = bottom[i] + heights[i]
            ax.text(i, top_y, f"{v:+.1f}M",
                    ha="center", va="bottom" if v >= 0 else "top", fontsize=8)
        ax.text(3, total, f"{total:+.1f}M",
                ha="center", va="bottom" if total >= 0 else "top",
                fontsize=9, fontweight="bold")
        ax.axhline(0, color="black", lw=0.6)
        ax.set_title(f"{cg} DALY change 1990-2023")
        ax.set_ylabel("DALYs (millions)")
        ax.tick_params(axis="x", rotation=20)

    fig.suptitle("Das Gupta decomposition of DALY change, Vietnam", fontsize=11)
    fig.tight_layout()
    save_fig(fig, "figure3_decomposition")
    plt.close(fig)


def figure_4(metrics, sdi_df):
    """SEA NCD-share trajectories + SDI vs NCD share scatter.
    Lim SS et al. Lancet 2018;392:2091-138 (SDI-expected method).
    """
    print("  Figure 4 - SEA NCD share comparison")
    fig, axes = plt.subplots(1, 2, figsize=(W2, W2 * 0.55))

    # NCD-share trajectories
    ax = axes[0]
    for country in SEA_COUNTRIES:
        sub = metrics[metrics["location_name"] == country].sort_values("year")
        disp = COUNTRY_DISPLAY.get(country, country)
        if country == "Viet Nam":
            ax.plot(sub["year"], sub["ncd_share_pct"],
                    color=PALETTE["Vietnam"], lw=2.5, label="Vietnam", zorder=10)
        else:
            ax.plot(sub["year"], sub["ncd_share_pct"],
                    color=PALETTE["neutral"], lw=1, alpha=0.7, label=disp)
    ax.axhline(50, color="black", ls=":", lw=1)
    ax.set_xlabel("Year")
    ax.set_ylabel("NCD share of total DALYs (%)")
    ax.set_title("NCD share trajectories, 11 SEA countries")
    ax.set_ylim(0, 100)
    ax.set_xlim(YEAR_MIN, YEAR_MAX)

    # End-of-line labels, pushed apart vertically to avoid overlap
    ends = []
    for country in SEA_COUNTRIES:
        sub = metrics[metrics["location_name"] == country].sort_values("year")
        if len(sub):
            ends.append((country, float(sub["ncd_share_pct"].iloc[-1])))
    ends.sort(key=lambda t: t[1])
    min_gap = 2.8  # percentage points
    adjusted = []
    for i, (c, y) in enumerate(ends):
        if i == 0:
            adjusted.append((c, y))
        else:
            prev_y = adjusted[-1][1]
            adjusted.append((c, max(y, prev_y + min_gap)))
    for country, y_text in adjusted:
        sub = metrics[metrics["location_name"] == country].sort_values("year")
        disp = COUNTRY_DISPLAY.get(country, country)
        y_end = float(sub["ncd_share_pct"].iloc[-1])
        col = PALETTE["Vietnam"] if country == "Viet Nam" else PALETTE["neutral"]
        ax.annotate(
            disp, xy=(YEAR_MAX, y_end), xytext=(YEAR_MAX + 0.8, y_text),
            fontsize=7, color=col, va="center",
            arrowprops=dict(arrowstyle="-", color=col, lw=0.3,
                            shrinkA=0, shrinkB=0, alpha=0.7),
        )
    ax.set_xlim(YEAR_MIN, YEAR_MAX + 7)

    # SDI vs NCD share scatter + quadratic fit excluding Vietnam
    ax = axes[1]
    merged = metrics.merge(
        sdi_df[["location_name", "year", "sdi"]],
        on=["location_name", "year"], how="left",
    )
    for country in SEA_COUNTRIES:
        sub = merged[merged["location_name"] == country].sort_values("year")
        if country == "Viet Nam":
            ax.plot(sub["sdi"], sub["ncd_share_pct"],
                    color=PALETTE["Vietnam"], lw=2.5, label="Vietnam", zorder=10)
            ax.scatter(sub["sdi"].iloc[0], sub["ncd_share_pct"].iloc[0],
                       color=PALETTE["Vietnam"], s=35, edgecolor="black",
                       zorder=11, label="1990")
            ax.scatter(sub["sdi"].iloc[-1], sub["ncd_share_pct"].iloc[-1],
                       color=PALETTE["Vietnam"], s=80, marker="*",
                       edgecolor="black", zorder=11, label="2023")
        else:
            ax.plot(sub["sdi"], sub["ncd_share_pct"],
                    color=PALETTE["neutral"], lw=0.9, alpha=0.55)

    fit_data = merged[(merged["location_name"] != "Viet Nam")
                      & merged["sdi"].notna() & merged["ncd_share_pct"].notna()]
    x = fit_data["sdi"].values
    y = fit_data["ncd_share_pct"].values
    if len(x) >= 3:
        coef = np.polyfit(x, y, 2)
        xline = np.linspace(x.min(), x.max(), 100)
        yline = np.polyval(coef, xline)
        ax.plot(xline, yline, color="black", ls="--", lw=1.3,
                label="SEA expected (quadratic, VN excluded)")
        # Annotate Vietnam 2023 observed/expected ratio
        vn23 = merged[(merged["location_name"] == "Viet Nam")
                      & (merged["year"] == 2023)]
        if not vn23.empty:
            sdi23 = float(vn23["sdi"].iloc[0])
            obs = float(vn23["ncd_share_pct"].iloc[0])
            exp = float(np.polyval(coef, sdi23))
            ratio = obs / exp if exp else np.nan
            ax.annotate(
                f"VN obs/exp 2023 = {ratio:.3f}",
                xy=(sdi23, obs), xytext=(sdi23 - 0.08, obs + 12),
                fontsize=8, color=PALETTE["Vietnam"],
                arrowprops=dict(arrowstyle="->", color=PALETTE["Vietnam"],
                                lw=0.8, shrinkA=0, shrinkB=2),
            )
    ax.set_xlabel("Socio-Demographic Index (SDI)")
    ax.set_ylabel("NCD share of total DALYs (%)")
    ax.set_title("SDI vs NCD share, SEA (Vietnam highlighted)")
    ax.set_ylim(0, 100)
    ax.legend(loc="lower right", frameon=False, fontsize=8)

    fig.tight_layout()
    save_fig(fig, "figure4_sea_comparison")
    plt.close(fig)


def figure_5(data):
    """Age-sex pyramid 1990 vs 2023, CMNN vs NCD burden."""
    print("  Figure 5 - Age-sex pyramid shift")
    q3 = data["q3"]
    age_groups = [
        "<5 years", "5-9 years", "10-14 years", "15-19 years", "20-24 years",
        "25-29 years", "30-34 years", "35-39 years", "40-44 years",
        "45-49 years", "50-54 years", "55-59 years", "60-64 years",
        "65-69 years", "70-74 years", "75-79 years", "80+ years",
    ]
    groups_map = {CMNN_LABEL: "CMNN", NCD_LABEL: "NCD"}
    sub = q3[
        (q3["measure_name"] == DALY_LABEL)
        & (q3["metric_name"] == "Number")
        & (q3["sex_name"].isin(["Male", "Female"]))
        & (q3["age_name"].isin(age_groups))
        & (q3["cause_name"].isin(groups_map.keys()))
        & (q3["year"].isin([1990, 2023]))
    ].copy()
    sub["cause_group"] = sub["cause_name"].map(groups_map)

    fig, axes = plt.subplots(1, 2, figsize=(W2, W2 * 0.55),
                             sharey=True, sharex=True)

    panels = {}
    global_xmax = 0.0
    for yr in [1990, 2023]:
        pv = sub[sub["year"] == yr].pivot_table(
            index="age_name", columns=["sex_name", "cause_group"],
            values="val", aggfunc="sum",
        ).reindex(age_groups)
        m_cmnn = -(pv[("Male", "CMNN")].values / 1e6)
        m_ncd = -(pv[("Male", "NCD")].values / 1e6)
        f_cmnn = pv[("Female", "CMNN")].values / 1e6
        f_ncd = pv[("Female", "NCD")].values / 1e6
        panels[yr] = (m_cmnn, m_ncd, f_cmnn, f_ncd)
        global_xmax = max(global_xmax, float(np.nanmax(np.abs(
            np.concatenate([m_cmnn + m_ncd, f_cmnn + f_ncd])))))

    for ax, yr in zip(axes, [1990, 2023]):
        m_cmnn, m_ncd, f_cmnn, f_ncd = panels[yr]
        ys = np.arange(len(age_groups))
        ax.barh(ys, m_cmnn, color=PALETTE["CMNN"], alpha=0.9,
                edgecolor="white", lw=0.4)
        ax.barh(ys, m_ncd, left=m_cmnn, color=PALETTE["NCD"], alpha=0.9,
                edgecolor="white", lw=0.4)
        ax.barh(ys, f_cmnn, color=PALETTE["CMNN"], alpha=0.9,
                edgecolor="white", lw=0.4)
        ax.barh(ys, f_ncd, left=f_cmnn, color=PALETTE["NCD"], alpha=0.9,
                edgecolor="white", lw=0.4)
        ax.axvline(0, color="black", lw=0.6)
        ax.set_yticks(ys)
        ax.set_yticklabels(age_groups)
        ax.set_title(f"{yr}")
        ax.set_xlabel("DALYs (millions)  -  male (left) / female (right)")
        ax.set_xlim(-global_xmax * 1.05, global_xmax * 1.05)
        ax.xaxis.set_major_formatter(
            mpl.ticker.FuncFormatter(lambda v, _: f"{abs(v):.1f}")
        )

    handles = [
        Patch(color=PALETTE["CMNN"], label="CMNN"),
        Patch(color=PALETTE["NCD"], label="NCD"),
    ]
    fig.legend(handles=handles, loc="upper center", ncol=2,
               frameon=False, bbox_to_anchor=(0.5, 1.02))
    fig.suptitle("Vietnam DALY burden by age, sex and cause group: 1990 vs 2023",
                 fontsize=11, y=1.06)
    fig.tight_layout()
    save_fig(fig, "figure5_age_sex_pyramid")
    plt.close(fig)


def section6_figures(data, metrics, decomp, sdi_df):
    print("\n" + "=" * 70)
    print("SECTION 6 - Figures")
    print("=" * 70)
    figure_1(data, metrics)
    figure_2(data)
    figure_3(decomp)
    figure_4(metrics, sdi_df)
    figure_5(data)


## SECTION 7 - SUMMARY TABLE

In [8]:
# =============================================================================
# SECTION 7 - SUMMARY TABLE
# =============================================================================

def section7_summary(data, metrics, trends):
    print("\n" + "=" * 70)
    print("SECTION 7 - Summary Table")
    print("=" * 70)

    q1_asr = data["q1_asr"]
    vn = q1_asr[
        (q1_asr["location_name"] == "Viet Nam")
        & (q1_asr["measure_name"] == DALY_LABEL)
        & (q1_asr["metric_name"] == "Rate")
        & (q1_asr["sex_name"] == "Both")
    ].copy()
    pvt = vn.pivot_table(index="year", columns="cause_group", values="val")

    years = [1990, 2000, 2010, 2023]
    aapc_by_group = (trends[trends["country"] == "Vietnam"]
                     .drop_duplicates("cause_group")
                     .set_index("cause_group")["aapc_pct_1990_2023"])
    vn_m = metrics[metrics["country"] == "Vietnam"].set_index("year")

    rows = []
    for g in ["CMNN", "NCD", "Injuries"]:
        r = {"metric": f"{g} age-std DALY rate (per 100k)"}
        for y in years:
            r[str(y)] = round(pvt.loc[y, g], 1) if y in pvt.index else np.nan
        r["AAPC_%"] = round(aapc_by_group.get(g, np.nan), 2)
        rows.append(r)

    for col, label in [
        ("cmnn_share_pct", "CMNN share of total DALYs (%)"),
        ("ncd_share_pct", "NCD share of total DALYs (%)"),
        ("injuries_share_pct", "Injuries share of total DALYs (%)"),
        ("cmnn_ncd_ratio", "CMNN/NCD ratio"),
    ]:
        r = {"metric": label}
        for y in years:
            r[str(y)] = (round(vn_m.loc[y, col],
                               3 if "ratio" in col else 2)
                         if y in vn_m.index else np.nan)
        r["AAPC_%"] = ""
        rows.append(r)

    tbl = pd.DataFrame(rows)
    tbl.to_csv(TAB / "table1_summary.csv", index=False)
    print(tbl.to_string(index=False))
    print("  --> tables/table1_summary.csv")
    return tbl


## Run pipeline

In [9]:
data = section1_load_clean()
metrics = section2_core_metrics(data)
trends = section3_trends(data)
decomp = section4_decomposition(data)
sea = section5_sea(data, metrics)
section6_figures(data, metrics, decomp, data['sdi'])
summary = section7_summary(data, metrics, trends)
print('DONE.')


SECTION 1 - Load & Clean Data
  query1 (level-1 SEA+Timor): 10,608 rows, 11 locations
  query2a (Vietnam CMNN detail): 1,632 rows, 8 causes
  query2b (Vietnam NCD detail):  2,550 rows, 13 causes
  query3 (Vietnam age-specific): 29,376 rows, ages=18
  query4 (Vietnam population): 1,836 rows


  SDI: 374 rows, locations=11



  --- Data quality ---
  q1: missing=0, year=[1990-2023], locations=11
  q2a: missing=0, year=[1990-2023], locations=1
  q2b: missing=0, year=[1990-2023], locations=1
  q3: missing=0, year=[1990-2023], locations=1
  q4: missing=0, year=[1990-2023], locations=1
  sdi: missing=0, year=[1990-2023], locations=11
  q1: age-std=2,992 rows, all-ages=6,256 rows
  q2a: age-std=544 rows, all-ages=1,088 rows
  q2b: age-std=850 rows, all-ages=1,700 rows


  --> cleaned files written to data/processed/

SECTION 2 - Core Metrics (cause composition shares)
  NCD share of total DALYs in 2023 by country:
    country  cmnn_share_pct  ncd_share_pct  injuries_share_pct  cmnn_ncd_ratio
     Brunei           9.716         81.263               9.021           0.120
  Singapore           9.445         79.284              11.271           0.119
   Malaysia          16.848         75.042               8.110           0.225
    Vietnam          14.740         70.667              14.592           0.209
   Thailand          17.086         68.376              14.537           0.250
  Indonesia          24.621         68.037               7.342           0.362
Philippines          25.213         67.343               7.444           0.374
    Myanmar          23.799         65.386              10.814           0.364
   Cambodia          24.998         63.829              11.173           0.392
Timor-Leste          30.643         59.583               9.774 

  Vietnam AAPC 1990-2023 (age-std DALY rate):
cause_group  aapc_pct_1990_2023     joinpoints  n_breakpoints
       CMNN               -4.63 1996;1999;2020              3
        NCD               -0.37      1995;2008              2
   Injuries               -1.13 1995;1997;2006              3
  --> tables/trend_results.csv (114 segment rows)

SECTION 4 - Das Gupta Decomposition (1990 -> 2023)
  Observed vs decomposed change in total DALYs (Vietnam):
cause_group  pop_size  age_structure  age_rate  total_decomp  observed_change direction
       CMNN   3247842       -1508711  -8184520      -6445388         -6445388  decrease
        NCD   6257108        6080731  -1710855      10626984         10626984  increase
  --> tables/decomposition_results.csv

SECTION 5 - SEA Comparison
  SEA countries ranked by NCD share of total DALYs, 2023:
    country  ncd_share_1990  ncd_share_2023  delta_ncd_share_pp  obs_vs_expected_ratio_2023
     Brunei          76.117          81.263               5.146  

  saved figures/figure1_overview.png + .svg
  Figure 2 - Top-15 cause heatmap (Vietnam)


  saved figures/figure2_heatmap.png + .svg
  Figure 3 - Decomposition waterfall


  saved figures/figure3_decomposition.png + .svg
  Figure 4 - SEA NCD share comparison


  saved figures/figure4_sea_comparison.png + .svg
  Figure 5 - Age-sex pyramid shift


  saved figures/figure5_age_sex_pyramid.png + .svg

SECTION 7 - Summary Table
                               metric      1990      2000      2010      2023 AAPC_%
    CMNN age-std DALY rate (per 100k) 13295.900  8549.000  5870.200  4022.100  -4.63
     NCD age-std DALY rate (per 100k) 21688.200 19960.400 19742.800 19282.800  -0.37
Injuries age-std DALY rate (per 100k)  5942.000  5409.900  4683.000  3981.800  -1.13
        CMNN share of total DALYs (%)    32.490    25.200    19.380    14.740       
         NCD share of total DALYs (%)    52.990    58.850    65.170    70.670       
    Injuries share of total DALYs (%)    14.520    15.950    15.460    14.590       
                       CMNN/NCD ratio     0.613     0.428     0.297     0.209       
  --> tables/table1_summary.csv
DONE.
